In [1]:
import pandas as pd
import numpy as np
import os
import re
import gc
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold
from scipy import stats
from statsmodels.stats.multitest import multipletests
import time

# paths
AA_GENO  = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint7_snp_encoded_012.csv"
AA_META  = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint2_metadata_sample_filtered.csv"
MANIFEST = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"
OUT_DIR  = r"C:\Users\user\Desktop\ai causal\FTND\african_amercian"

os.makedirs(OUT_DIR, exist_ok=True)
print("Imports done. Output dir:", OUT_DIR)

Imports done. Output dir: C:\Users\user\Desktop\ai causal\FTND\african_amercian


In [2]:
meta_df = pd.read_csv(AA_META)
meta_df["sample_id"] = meta_df["sample_id"].astype(str)

# filter to smokers only
smokers = meta_df[meta_df["smoking_status"] == "Smoker"].copy()
smokers["ftnd"] = pd.to_numeric(smokers["ftnd"], errors="coerce")

# drop any remaining missing FTND
smokers = smokers.dropna(subset=["ftnd"])
smokers = smokers[smokers["ftnd"] >= 0]  # remove any -9 codes

print("Total AA samples:", len(meta_df))
print("Smokers with valid FTND:", len(smokers))
print("FTND distribution:")
print(smokers["ftnd"].describe())
print("FTND value counts:")
print(smokers["ftnd"].value_counts().sort_index())

smoker_ids = smokers["sample_id"].tolist()

Total AA samples: 3348
Smokers with valid FTND: 1631
FTND distribution:
count    1631.000000
mean        8.429185
std         1.481495
min         0.000000
25%         9.000000
50%         9.000000
75%         9.000000
max        10.000000
Name: ftnd, dtype: float64
FTND value counts:
ftnd
0        4
1        4
2        9
3       18
4       28
5       38
6       57
7       99
8      127
9     1151
10      96
Name: count, dtype: int64


In [3]:
def strip_suffix(pid):
    return re.sub(r'_\d+$', '', pid)

# load manifest for chromosome info
manifest_df = pd.read_csv(MANIFEST, skiprows=7, low_memory=False)
manifest_df["core_name"] = manifest_df["IlmnID"].map(strip_suffix)
non_autosomal = {"X", "Y", "XY", "MT", "0"}
sex_linked_cores = set(
    manifest_df.loc[manifest_df["Chr"].isin(non_autosomal), "core_name"]
)

# load genotype matrix
encoded_df = pd.read_csv(AA_GENO)
probe_id_array = encoded_df["probe_id"].to_numpy()
all_sample_ids = encoded_df.columns[1:].tolist()

# filter columns to smokers only
smoker_id_set = set(smoker_ids)
smoker_cols = [sid for sid in all_sample_ids if sid in smoker_id_set]
print("Smoker columns found in genotype file:", len(smoker_cols))

# align smokers metadata to genotype column order
smokers_aligned = smokers.set_index("sample_id").reindex(smoker_cols).reset_index()
Y_ftnd = smokers_aligned["ftnd"].values.astype(np.float64)
age_std = ((smokers_aligned["age"].astype(float) - smokers_aligned["age"].astype(float).mean())
           / smokers_aligned["age"].astype(float).std()).values
gender_binary = (smokers_aligned["gender"] == "Male").astype(np.float64).values

print("Y_ftnd shape:", Y_ftnd.shape)
print("Y_ftnd mean:", Y_ftnd.mean().round(3))

# filter to autosomal probes
keep_mask = ~np.array([strip_suffix(p) in sex_linked_cores for p in probe_id_array])

# extract smoker columns as numpy array
X_snp = encoded_df[smoker_cols].to_numpy(dtype=np.int8)[keep_mask]
X_auto = X_snp.T  # smokers x SNPs
probe_ids_auto = probe_id_array[keep_mask]

del encoded_df, X_snp
gc.collect()

print("X_auto shape:", X_auto.shape)  # expect (1631, 233610)

Smoker columns found in genotype file: 1631
Y_ftnd shape: (1631,)
Y_ftnd mean: 8.429
X_auto shape: (1631, 233610)


In [4]:
# EIGENSTRAT standardization
X_float = X_auto.astype(np.float32)
del X_auto
gc.collect()

p = X_float.mean(axis=0) / 2
denom = np.sqrt(2 * p * (1 - p))
valid_mask = denom > 1e-8
print("Monomorphic excluded:", (~valid_mask).sum())
print("Informative retained:", valid_mask.sum())

X_std = (X_float[:, valid_mask] - 2 * p[valid_mask]) / denom[valid_mask]
probe_ids_valid = probe_ids_auto[valid_mask]

del X_float
gc.collect()

print("X_std shape:", X_std.shape)

# PCA top 10
pca = PCA(n_components=10, random_state=42, copy=False, svd_solver='randomized')
pcs = pca.fit_transform(X_std)
print("Explained variance ratio:", pca.explained_variance_ratio_)

# confounder matrix: [PC1..10, age, gender]
X_conf = np.hstack([
    pcs,
    age_std.reshape(-1, 1),
    gender_binary.reshape(-1, 1)
]).astype(np.float64)

print("X_conf shape:", X_conf.shape)  # expect (1631, 12)

Monomorphic excluded: 106146
Informative retained: 127464
X_std shape: (1631, 127464)
Explained variance ratio: [0.00717025 0.00188095 0.00178024 0.00172667 0.00171314 0.0015741
 0.0015012  0.00145438 0.0014028  0.00138412]
X_conf shape: (1631, 12)


In [6]:
def doubleml_scan(X_snps, Y, X_conf, n_folds=5, random_state=42):
    n, n_snps = X_snps.shape
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    D_resid = np.zeros_like(X_snps)
    Y_resid = np.zeros(n)
    Xc = np.column_stack([np.ones(n), X_conf])

    for train_idx, test_idx in kf.split(Xc):
        Xc_tr, Xc_te = Xc[train_idx], Xc[test_idx]
        coef_Y = np.linalg.lstsq(Xc_tr, Y[train_idx], rcond=None)[0]
        Y_resid[test_idx] = Y[test_idx] - Xc_te @ coef_Y
        coef_D = np.linalg.lstsq(Xc_tr, X_snps[train_idx], rcond=None)[0]
        D_resid[test_idx] = X_snps[test_idx] - Xc_te @ coef_D

    Yr = Y_resid - Y_resid.mean()
    Dr = D_resid - D_resid.mean(axis=0)
    del D_resid

    sum_DY = (Dr * Yr[:, None]).sum(axis=0)
    sum_DD = (Dr ** 2).sum(axis=0)
    sum_YY = (Yr ** 2).sum()

    theta = sum_DY / sum_DD
    ssr = sum_YY - (sum_DY ** 2) / sum_DD
    sigma2 = ssr / (n - 2)
    se = np.sqrt(sigma2 / sum_DD)
    t_stat = theta / se
    pvals = 2 * (1 - stats.t.cdf(np.abs(t_stat), df=n - 2))

    return theta, pvals

print("Running single test iteration...")
start = time.time()
theta_test, pvals_test = doubleml_scan(X_std, Y_ftnd, X_conf, random_state=0)
print(f"Completed in {time.time()-start:.1f}s")

for thresh in [0.01, 0.001, 0.0001]:
    print(f"Raw p < {thresh}: {(pvals_test < thresh).sum()} SNPs")

Running single test iteration...
Completed in 34.6s
Raw p < 0.01: 3034 SNPs
Raw p < 0.001: 1267 SNPs
Raw p < 0.0001: 512 SNPs


In [7]:
n_repeats = 30
threshold = 0.0001
n_snps = X_std.shape[1]

significant_counts = np.zeros(n_snps, dtype=np.int32)

print(f"Running {n_repeats} stability repeats (p<{threshold})...")
start = time.time()

for rep in range(n_repeats):
    _, pvals_rep = doubleml_scan(X_std, Y_ftnd, X_conf, random_state=rep)
    significant_counts += (pvals_rep < threshold).astype(np.int32)
    if (rep + 1) % 5 == 0:
        print(f"  Repeat {rep+1}/{n_repeats} — {(time.time()-start)/60:.1f} min")

stability_fraction = significant_counts / n_repeats

stability_df = pd.DataFrame({
    "probe_id": probe_ids_valid,
    "stability_fraction": stability_fraction,
    "n_significant_repeats": significant_counts
}).sort_values("stability_fraction", ascending=False)

stability_df.to_csv(os.path.join(OUT_DIR, "ftnd_stability_results.csv"), index=False)

print(f"\nTotal: {(time.time()-start)/60:.1f} min")
for thresh in [0.5, 0.7, 0.8, 0.9, 1.0]:
    print(f"  >= {thresh:.0%} stability: {(stability_fraction >= thresh).sum()} SNPs")

Running 30 stability repeats (p<0.0001)...
  Repeat 5/30 — 2.8 min
  Repeat 10/30 — 5.6 min
  Repeat 15/30 — 8.3 min
  Repeat 20/30 — 10.9 min
  Repeat 25/30 — 13.6 min
  Repeat 30/30 — 16.3 min

Total: 16.3 min
  >= 50% stability: 532 SNPs
  >= 70% stability: 511 SNPs
  >= 80% stability: 503 SNPs
  >= 90% stability: 491 SNPs
  >= 100% stability: 428 SNPs


In [8]:
# check if we should use stricter threshold
print("Current (p<0.0001, 100% stability): 428 SNPs")
print("\nLet's check what p<0.00001 gives in one iteration:")

start = time.time()
_, pvals_strict = doubleml_scan(X_std, Y_ftnd, X_conf, random_state=0)
print(f"Completed in {time.time()-start:.1f}s")

for thresh in [0.00001, 0.000001, 0.0000001]:
    print(f"Raw p < {thresh}: {(pvals_strict < thresh).sum()} SNPs")

# also check top SNPs by stability fraction
stability_df = pd.read_csv(os.path.join(OUT_DIR, "ftnd_stability_results.csv"))
print("\nTop 20 most stable SNPs:")
print(stability_df.head(20)[["probe_id", "stability_fraction", "n_significant_repeats"]])

Current (p<0.0001, 100% stability): 428 SNPs

Let's check what p<0.00001 gives in one iteration:
Completed in 24.2s
Raw p < 1e-05: 285 SNPs
Raw p < 1e-06: 158 SNPs
Raw p < 1e-07: 91 SNPs

Top 20 most stable SNPs:
                       probe_id  stability_fraction  n_significant_repeats
0     exm26794-0_B_R_1921571564                 1.0                     30
1    exm326427-0_B_R_1922217674                 1.0                     30
2    exm223412-0_B_F_1918813765                 1.0                     30
3    exm523594-0_T_F_1921962613                 1.0                     30
4    exm551051-0_T_F_1921880849                 1.0                     30
5    exm294694-0_B_F_1922223929                 1.0                     30
6    exm466812-0_T_R_1921194521                 1.0                     30
7   exm1189236-0_B_R_1922835354                 1.0                     30
8   exm1189432-0_T_R_1922833074                 1.0                     30
9   exm1191937-0_B_F_1922855101      

In [9]:
n_repeats = 30
threshold = 1e-6
n_snps = X_std.shape[1]

significant_counts_v2 = np.zeros(n_snps, dtype=np.int32)

print(f"Running {n_repeats} stability repeats (p<{threshold})...")
start = time.time()

for rep in range(n_repeats):
    _, pvals_rep = doubleml_scan(X_std, Y_ftnd, X_conf, random_state=rep)
    significant_counts_v2 += (pvals_rep < threshold).astype(np.int32)
    if (rep + 1) % 5 == 0:
        print(f"  Repeat {rep+1}/{n_repeats} — {(time.time()-start)/60:.1f} min")

stability_fraction_v2 = significant_counts_v2 / n_repeats

stability_df_v2 = pd.DataFrame({
    "probe_id": probe_ids_valid,
    "stability_fraction": stability_fraction_v2,
    "n_significant_repeats": significant_counts_v2
}).sort_values("stability_fraction", ascending=False)

stability_df_v2.to_csv(os.path.join(OUT_DIR, "ftnd_stability_results_v2.csv"), index=False)

print(f"\nTotal: {(time.time()-start)/60:.1f} min")
for thresh in [0.5, 0.7, 0.8, 0.9, 1.0]:
    print(f"  >= {thresh:.0%} stability: {(stability_fraction_v2 >= thresh).sum()} SNPs")

Running 30 stability repeats (p<1e-06)...
  Repeat 5/30 — 2.8 min
  Repeat 10/30 — 5.7 min
  Repeat 15/30 — 8.5 min
  Repeat 20/30 — 11.0 min
  Repeat 25/30 — 13.9 min
  Repeat 30/30 — 16.8 min

Total: 16.8 min
  >= 50% stability: 169 SNPs
  >= 70% stability: 149 SNPs
  >= 80% stability: 148 SNPs
  >= 90% stability: 142 SNPs
  >= 100% stability: 117 SNPs


In [10]:
import re

shortlist_ftnd = stability_df_v2[stability_df_v2["stability_fraction"] == 1.0].copy()
shortlist_ftnd["core_name"] = shortlist_ftnd["probe_id"].map(strip_suffix)

pos_lookup = manifest_df.set_index("core_name")[["Chr", "MapInfo"]]
shortlist_ftnd = shortlist_ftnd.merge(pos_lookup, on="core_name", how="left")
shortlist_ftnd = shortlist_ftnd[~shortlist_ftnd["Chr"].isin(non_autosomal)].copy()

print("100% stable SNPs:", len(shortlist_ftnd))

# load genotype vectors for LD pruning
encoded_df_ld = pd.read_csv(AA_GENO)
probe_rows = encoded_df_ld[encoded_df_ld["probe_id"].isin(set(shortlist_ftnd["probe_id"]))].copy()
probe_rows = probe_rows.set_index("probe_id").reindex(shortlist_ftnd["probe_id"].tolist())
X_shortlist = probe_rows.to_numpy(dtype=np.float64).T
probe_id_to_idx = {pid: i for i, pid in enumerate(shortlist_ftnd["probe_id"].tolist())}

del encoded_df_ld, probe_rows
gc.collect()

def get_geno(pid):
    return X_shortlist[:, probe_id_to_idx[pid]]

def greedy_ld_prune(df, r2_thresh=0.2, window_bp=1_000_000):
    sorted_df = df.sort_values("stability_fraction", ascending=False).reset_index(drop=True)
    retained = []
    removed = set()
    for i, row_i in sorted_df.iterrows():
        pid_i = row_i["probe_id"]
        if pid_i in removed:
            continue
        retained.append(pid_i)
        g_i = get_geno(pid_i)
        for j, row_j in sorted_df.iloc[i+1:].iterrows():
            pid_j = row_j["probe_id"]
            if pid_j in removed:
                continue
            if row_j["Chr"] != row_i["Chr"]:
                continue
            if abs(row_j["MapInfo"] - row_i["MapInfo"]) > window_bp:
                continue
            r2 = np.corrcoef(g_i, get_geno(pid_j))[0, 1] ** 2
            if r2 > r2_thresh:
                removed.add(pid_j)
    return retained

retained = greedy_ld_prune(shortlist_ftnd, r2_thresh=0.2)
shortlist_pruned_ftnd = shortlist_ftnd[
    shortlist_ftnd["probe_id"].isin(retained)
].copy().sort_values(["Chr", "MapInfo"]).reset_index(drop=True)

print(f"After LD pruning: {len(shortlist_pruned_ftnd)} SNPs")
shortlist_pruned_ftnd.to_csv(os.path.join(OUT_DIR, "ftnd_shortlist_ld_pruned.csv"), index=False)
print(shortlist_pruned_ftnd[["probe_id", "Chr", "MapInfo"]].to_string())

100% stable SNPs: 117
After LD pruning: 112 SNPs
                             probe_id Chr      MapInfo
0            exm1790-0_B_F_1921371183   1    1149109.0
1           exm43011-0_B_F_1921441001   1   35250486.0
2           exm45382-0_T_R_1921413452   1   36904409.0
3           exm63579-0_T_R_1921559958   1   58999918.0
4          exm119079-0_T_F_1921575913   1  162725022.0
5          exm120679-0_T_F_1921471407   1  167780083.0
6          exm131374-0_B_F_1919141167   1  185985176.0
7         exm2277017-0_T_R_1989215336   1  202399880.0
8          exm144975-0_B_R_1919114304   1  207793411.0
9          exm150738-0_B_R_1919139270   1  221879742.0
10         exm151723-0_B_R_1921475100   1  223568054.0
11         exm161959-0_T_R_1919158388   1  236730158.0
12         exm162451-0_T_R_1921466416   1  237054460.0
13         exm806278-0_T_R_1921036879  10    3185622.0
14         exm823703-0_B_R_1920989983  10   50340187.0
15         exm826689-0_T_R_1921049687  10   61829951.0
16         exm83

In [11]:
print("SNPs per chromosome after LD pruning:")
print(shortlist_pruned_ftnd["Chr"].value_counts().sort_index())

print("\nTotal SNPs:", len(shortlist_pruned_ftnd))
print("\nChromosomes represented:", sorted(shortlist_pruned_ftnd["Chr"].unique()))

SNPs per chromosome after LD pruning:
Chr
1     13
10     6
11     7
12     6
13     5
14     2
15     4
16     6
17     7
18     1
19     5
2      7
20     3
21     3
22     2
3     12
4      5
5      3
6      8
7      2
8      1
9      4
Name: count, dtype: int64

Total SNPs: 112

Chromosomes represented: ['1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '3', '4', '5', '6', '7', '8', '9']


In [12]:
known_regions = {
    "CHRNA5/CHRNA3/CHRNB4": ("15", 78_800_000, 79_100_000),
    "CYP2A6": ("19", 41_347_000, 41_400_000),
    "CHRNA4": ("20", 61_900_000, 62_000_000),
}

print("=== KNOWN DEPENDENCE LOCI CHECK ===")
for gene, (chrom, start, end) in known_regions.items():
    hits = shortlist_pruned_ftnd[
        (shortlist_pruned_ftnd["Chr"].astype(str) == chrom) &
        (shortlist_pruned_ftnd["MapInfo"] >= start) &
        (shortlist_pruned_ftnd["MapInfo"] <= end)
    ]
    if len(hits) > 0:
        print(f"✓ HIT: {gene}")
        for _, row in hits.iterrows():
            print(f"    {row['probe_id']} pos={int(row['MapInfo']):,}")
    else:
        print(f"✗ No hit: {gene}")

# also check wider window (5Mb) around these regions
print("\n=== WIDER CHECK (5Mb window) ===")
for gene, (chrom, start, end) in known_regions.items():
    hits = shortlist_pruned_ftnd[
        (shortlist_pruned_ftnd["Chr"].astype(str) == chrom) &
        (shortlist_pruned_ftnd["MapInfo"] >= start - 5_000_000) &
        (shortlist_pruned_ftnd["MapInfo"] <= end + 5_000_000)
    ]
    if len(hits) > 0:
        print(f"✓ NEAR: {gene} (within 5Mb)")
        for _, row in hits.iterrows():
            print(f"    {row['probe_id']} pos={int(row['MapInfo']):,} dist={min(abs(int(row['MapInfo'])-start), abs(int(row['MapInfo'])-end)):,}bp")
    else:
        print(f"✗ Nothing within 5Mb: {gene}")

=== KNOWN DEPENDENCE LOCI CHECK ===
✗ No hit: CHRNA5/CHRNA3/CHRNB4
✗ No hit: CYP2A6
✗ No hit: CHRNA4

=== WIDER CHECK (5Mb window) ===
✗ Nothing within 5Mb: CHRNA5/CHRNA3/CHRNB4
✓ NEAR: CYP2A6 (within 5Mb)
    exm1471153-0_B_R_1923230572 pos=41,928,086 dist=528,086bp
✗ Nothing within 5Mb: CHRNA4


In [14]:
# check where rs16969968 fell in the FTND stability results
known_probes = {
    "rs16969968 (CHRNA5)": "exm1180599-0_T_F_1922812583",
    "rs4833101 (CHRNB4)":  "exm-rs8034191-131_B_F_1990485548",
    "rs3025343 (DRD2)":    "exm-rs3025343-131_T_F_1990491231"
}

stability_df_v2 = pd.read_csv(os.path.join(OUT_DIR, "ftnd_stability_results_v2.csv"))

print("=== KNOWN SNP STATUS IN FTND PIPELINE ===\n")
for name, probe_id in known_probes.items():
    # check if in valid SNP set
    in_valid = probe_id in probe_ids_valid
    
    # check stability
    row = stability_df_v2[stability_df_v2["probe_id"] == probe_id]
    
    if len(row) > 0:
        stab = row["stability_fraction"].values[0]
        n_sig = row["n_significant_repeats"].values[0]
        print(f"{name}")
        print(f"  In informative SNP set: {in_valid}")
        print(f"  Stability: {stab:.3f} ({n_sig}/30 repeats)")
        print(f"  In final shortlist: {probe_id in shortlist_pruned_ftnd['probe_id'].values}")
    else:
        print(f"{name}: not in stability results")
        print(f"  In informative SNP set: {in_valid}")
    print()

=== KNOWN SNP STATUS IN FTND PIPELINE ===

rs16969968 (CHRNA5)
  In informative SNP set: True
  Stability: 0.000 (0/30 repeats)
  In final shortlist: False

rs4833101 (CHRNB4)
  In informative SNP set: True
  Stability: 0.000 (0/30 repeats)
  In final shortlist: False

rs3025343 (DRD2)
  In informative SNP set: True
  Stability: 0.000 (0/30 repeats)
  In final shortlist: False



In [15]:
print("=== HSI distribution (smokers only) ===")
smokers["hsi"] = pd.to_numeric(smokers["hsi"], errors="coerce")
smokers_hsi = smokers[smokers["hsi"] >= 0]
print(smokers_hsi["hsi"].describe())
print(smokers_hsi["hsi"].value_counts().sort_index())

print("\n=== CPD distribution (smokers only) ===")
smokers["cpd"] = pd.to_numeric(smokers["cpd"], errors="coerce")
smokers_cpd = smokers[smokers["cpd"] >= 0]
print(smokers_cpd["cpd"].describe())
print(smokers_cpd["cpd"].value_counts().sort_index())

=== HSI distribution (smokers only) ===
count    1631.000000
mean        4.762109
std         0.845667
min         0.000000
25%         5.000000
50%         5.000000
75%         5.000000
max         6.000000
Name: hsi, dtype: float64
hsi
0       8
1       8
2      33
3      86
4     167
5    1207
6     122
Name: count, dtype: int64

=== CPD distribution (smokers only) ===
count    1631.000000
mean       26.354997
std         6.507073
min         1.000000
25%        20.000000
50%        27.000000
75%        30.000000
max        60.000000
Name: cpd, dtype: float64
cpd
1       1
3       1
5       4
7       3
8       5
10     18
12      1
13      3
14      1
15      9
16      1
18      2
19      1
20    488
21     10
22     11
23      4
24      4
25    239
26      6
27      7
28      7
30    653
32      1
34      1
35     36
36      2
40    107
45      1
50      2
60      2
Name: count, dtype: int64


In [16]:
# build CPD outcome (smokers only, already filtered)
smokers["cpd"] = pd.to_numeric(smokers["cpd"], errors="coerce")
smokers_cpd = smokers[smokers["cpd"] >= 0].copy()

# realign to genotype column order
smokers_cpd_aligned = smokers_cpd.set_index("sample_id").reindex(smoker_cols).reset_index()
valid_cpd_mask = smokers_cpd_aligned["cpd"].notna()

# filter samples with valid CPD
smoker_cols_cpd = [s for s, v in zip(smoker_cols, valid_cpd_mask) if v]
Y_cpd = smokers_cpd_aligned.loc[valid_cpd_mask, "cpd"].values.astype(np.float64)
age_std_cpd = ((smokers_cpd_aligned.loc[valid_cpd_mask, "age"].astype(float) - 
                smokers_cpd_aligned.loc[valid_cpd_mask, "age"].astype(float).mean()) /
               smokers_cpd_aligned.loc[valid_cpd_mask, "age"].astype(float).std()).values
gender_binary_cpd = (smokers_cpd_aligned.loc[valid_cpd_mask, "gender"] == "Male").astype(np.float64).values

print("Samples with valid CPD:", len(Y_cpd))
print("CPD mean:", Y_cpd.mean().round(2), "std:", Y_cpd.std().round(2))

# rebuild X_std for CPD samples
encoded_df_cpd = pd.read_csv(AA_GENO)
X_snp_cpd = encoded_df_cpd[smoker_cols_cpd].to_numpy(dtype=np.int8)[keep_mask]
X_auto_cpd = X_snp_cpd.T
del encoded_df_cpd, X_snp_cpd
gc.collect()

# standardize
X_float_cpd = X_auto_cpd.astype(np.float32)
del X_auto_cpd
gc.collect()

p_cpd = X_float_cpd.mean(axis=0) / 2
denom_cpd = np.sqrt(2 * p_cpd * (1 - p_cpd))
valid_mask_cpd = denom_cpd > 1e-8
X_std_cpd = (X_float_cpd[:, valid_mask_cpd] - 2 * p_cpd[valid_mask_cpd]) / denom_cpd[valid_mask_cpd]
probe_ids_valid_cpd = probe_ids_auto[valid_mask_cpd]
del X_float_cpd
gc.collect()

print("X_std_cpd shape:", X_std_cpd.shape)

# PCA + confounders
pca_cpd = PCA(n_components=10, random_state=42, copy=False, svd_solver='randomized')
pcs_cpd = pca_cpd.fit_transform(X_std_cpd)

X_conf_cpd = np.hstack([
    pcs_cpd,
    age_std_cpd.reshape(-1, 1),
    gender_binary_cpd.reshape(-1, 1)
]).astype(np.float64)

print("X_conf_cpd shape:", X_conf_cpd.shape)

# single test iteration
print("\nRunning single test iteration for CPD...")
start = time.time()
theta_cpd, pvals_cpd = doubleml_scan(X_std_cpd, Y_cpd, X_conf_cpd, random_state=0)
print(f"Completed in {time.time()-start:.1f}s")

for thresh in [0.001, 0.0001, 1e-5, 1e-6]:
    print(f"Raw p < {thresh}: {(pvals_cpd < thresh).sum()} SNPs")

Samples with valid CPD: 1631
CPD mean: 26.35 std: 6.51
X_std_cpd shape: (1631, 127464)
X_conf_cpd shape: (1631, 12)

Running single test iteration for CPD...
Completed in 22.6s
Raw p < 0.001: 352 SNPs
Raw p < 0.0001: 95 SNPs
Raw p < 1e-05: 44 SNPs
Raw p < 1e-06: 31 SNPs


In [18]:
n_repeats = 30
threshold = 0.0001
n_snps_cpd = X_std_cpd.shape[1]

significant_counts_cpd = np.zeros(n_snps_cpd, dtype=np.int32)

print(f"Running {n_repeats} stability repeats (p<{threshold}) for CPD...")
start = time.time()

for rep in range(n_repeats):
    _, pvals_rep = doubleml_scan(X_std_cpd, Y_cpd, X_conf_cpd, random_state=rep)
    significant_counts_cpd += (pvals_rep < threshold).astype(np.int32)
    if (rep + 1) % 5 == 0:
        print(f"  Repeat {rep+1}/{n_repeats} — {(time.time()-start)/60:.1f} min")

stability_fraction_cpd = significant_counts_cpd / n_repeats

stability_df_cpd = pd.DataFrame({
    "probe_id": probe_ids_valid_cpd,
    "stability_fraction": stability_fraction_cpd,
    "n_significant_repeats": significant_counts_cpd
}).sort_values("stability_fraction", ascending=False)

stability_df_cpd.to_csv(os.path.join(OUT_DIR, "cpd_stability_results.csv"), index=False)

print(f"\nTotal: {(time.time()-start)/60:.1f} min")
for thresh in [0.5, 0.7, 0.8, 0.9, 1.0]:
    print(f"  >= {thresh:.0%} stability: {(stability_fraction_cpd >= thresh).sum()} SNPs")

Running 30 stability repeats (p<0.0001) for CPD...
  Repeat 5/30 — 2.1 min
  Repeat 10/30 — 4.0 min
  Repeat 15/30 — 5.9 min
  Repeat 20/30 — 7.8 min
  Repeat 25/30 — 9.8 min
  Repeat 30/30 — 11.7 min

Total: 11.7 min
  >= 50% stability: 185 SNPs
  >= 70% stability: 175 SNPs
  >= 80% stability: 154 SNPs
  >= 90% stability: 142 SNPs
  >= 100% stability: 60 SNPs


In [19]:
shortlist_cpd = stability_df_cpd[stability_df_cpd["stability_fraction"] == 1.0].copy()
shortlist_cpd["core_name"] = shortlist_cpd["probe_id"].map(strip_suffix)

pos_lookup_manifest = manifest_df.set_index("core_name")[["Chr", "MapInfo"]]
shortlist_cpd = shortlist_cpd.merge(pos_lookup_manifest, on="core_name", how="left")
shortlist_cpd = shortlist_cpd[~shortlist_cpd["Chr"].isin(non_autosomal)].copy()
print("100% stable SNPs:", len(shortlist_cpd))

# load genotype vectors for LD pruning (use smoker columns)
encoded_df_ld = pd.read_csv(AA_GENO)
probe_rows = encoded_df_ld[encoded_df_ld["probe_id"].isin(set(shortlist_cpd["probe_id"]))].copy()
probe_rows = probe_rows.set_index("probe_id").reindex(shortlist_cpd["probe_id"].tolist())
X_shortlist_cpd = probe_rows[smoker_cols_cpd].to_numpy(dtype=np.float64).T
probe_id_to_idx_cpd = {pid: i for i, pid in enumerate(shortlist_cpd["probe_id"].tolist())}
del encoded_df_ld, probe_rows
gc.collect()

def get_geno_cpd(pid):
    return X_shortlist_cpd[:, probe_id_to_idx_cpd[pid]]

def greedy_ld_prune(df, r2_thresh=0.2, window_bp=1_000_000):
    sorted_df = df.sort_values("stability_fraction", ascending=False).reset_index(drop=True)
    retained = []
    removed = set()
    for i, row_i in sorted_df.iterrows():
        pid_i = row_i["probe_id"]
        if pid_i in removed:
            continue
        retained.append(pid_i)
        g_i = get_geno_cpd(pid_i)
        for j, row_j in sorted_df.iloc[i+1:].iterrows():
            pid_j = row_j["probe_id"]
            if pid_j in removed:
                continue
            if row_j["Chr"] != row_i["Chr"]:
                continue
            if abs(row_j["MapInfo"] - row_i["MapInfo"]) > window_bp:
                continue
            r2 = np.corrcoef(g_i, get_geno_cpd(pid_j))[0, 1] ** 2
            if r2 > r2_thresh:
                removed.add(pid_j)
    return retained

retained_cpd = greedy_ld_prune(shortlist_cpd, r2_thresh=0.2)
shortlist_pruned_cpd = shortlist_cpd[
    shortlist_cpd["probe_id"].isin(retained_cpd)
].copy().sort_values(["Chr", "MapInfo"]).reset_index(drop=True)

print(f"After LD pruning: {len(shortlist_pruned_cpd)} SNPs")
shortlist_pruned_cpd.to_csv(os.path.join(OUT_DIR, "cpd_shortlist_ld_pruned.csv"), index=False)

# check known loci
print("\n=== KNOWN DEPENDENCE LOCI CHECK ===")
known_regions = {
    "CHRNA5/CHRNA3/CHRNB4": ("15", 78_800_000, 79_100_000),
    "CYP2A6": ("19", 41_347_000, 41_400_000),
    "CHRNA4": ("20", 61_900_000, 62_000_000),
}
for gene, (chrom, start, end) in known_regions.items():
    hits = shortlist_pruned_cpd[
        (shortlist_pruned_cpd["Chr"].astype(str) == chrom) &
        (shortlist_pruned_cpd["MapInfo"] >= start - 2_000_000) &
        (shortlist_pruned_cpd["MapInfo"] <= end + 2_000_000)
    ]
    if len(hits) > 0:
        print(f"✓ HIT/NEAR: {gene}")
        for _, row in hits.iterrows():
            dist = min(abs(int(row['MapInfo'])-start), abs(int(row['MapInfo'])-end))
            print(f"    {row['probe_id']} pos={int(row['MapInfo']):,} dist={dist:,}bp")
    else:
        print(f"✗ No hit: {gene}")

100% stable SNPs: 60
After LD pruning: 56 SNPs

=== KNOWN DEPENDENCE LOCI CHECK ===
✗ No hit: CHRNA5/CHRNA3/CHRNB4
✗ No hit: CYP2A6
✗ No hit: CHRNA4


In [2]:
import pandas as pd
import numpy as np
import json
import os
import re

OUT_DIR = r"C:\Users\user\Desktop\ai causal\FTND\african_amercian"
AA_GENO = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint7_snp_encoded_012.csv"
AA_META = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint2_metadata_sample_filtered.csv"
MANIFEST = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"

# reload metadata — smokers only with valid CPD
meta_df = pd.read_csv(AA_META)
meta_df["sample_id"] = meta_df["sample_id"].astype(str)
smokers = meta_df[meta_df["smoking_status"] == "Smoker"].copy()
smokers["cpd"] = pd.to_numeric(smokers["cpd"], errors="coerce")
smokers = smokers[smokers["cpd"] >= 0].dropna(subset=["cpd"])

# get genotype column order
encoded_df = pd.read_csv(AA_GENO)
all_sample_ids = encoded_df.columns[1:].tolist()
smoker_id_set = set(smokers["sample_id"].tolist())
smoker_cols_cpd = [s for s in all_sample_ids if s in smoker_id_set]

# align metadata to genotype order
smokers_aligned = smokers.set_index("sample_id").reindex(smoker_cols_cpd).reset_index()
Y_cpd = smokers_aligned["cpd"].values.astype(np.float64)

print("Smokers with valid CPD:", len(smoker_cols_cpd))
print("Y_cpd mean:", Y_cpd.mean().round(2))

# load shortlist and check perfect correlations
shortlist_pruned_cpd = pd.read_csv(os.path.join(OUT_DIR, "cpd_shortlist_ld_pruned.csv"))
pruned_ids_cpd = shortlist_pruned_cpd["probe_id"].tolist()

probe_rows = encoded_df[encoded_df["probe_id"].isin(set(pruned_ids_cpd))].copy()
probe_rows = probe_rows.set_index("probe_id").reindex(pruned_ids_cpd)
X_check = probe_rows[smoker_cols_cpd].to_numpy(dtype=np.float64).T

corr_matrix = np.corrcoef(X_check.T)
to_remove = set()
for i in range(len(pruned_ids_cpd)):
    for j in range(i+1, len(pruned_ids_cpd)):
        if abs(corr_matrix[i,j]) > 0.99 and pruned_ids_cpd[j] not in to_remove:
            to_remove.add(pruned_ids_cpd[j])

print(f"Perfectly correlated SNPs to remove: {len(to_remove)}")

shortlist_final_cpd = shortlist_pruned_cpd[
    ~shortlist_pruned_cpd["probe_id"].isin(to_remove)
].copy()
print(f"Final shortlist: {len(shortlist_final_cpd)} SNPs")
shortlist_final_cpd.to_csv(os.path.join(OUT_DIR, "cpd_shortlist_final.csv"), index=False)

# rebuild PC input
pruned_ids_final = shortlist_final_cpd["probe_id"].tolist()
col_names_cpd = pruned_ids_final + ["CPD"]

probe_rows_final = encoded_df[encoded_df["probe_id"].isin(set(pruned_ids_final))].copy()
probe_rows_final = probe_rows_final.set_index("probe_id").reindex(pruned_ids_final)
X_pc_cpd = probe_rows_final[smoker_cols_cpd].to_numpy(dtype=np.float64).T
X_pc_full_cpd = np.hstack([X_pc_cpd, Y_cpd.reshape(-1, 1)])

print("PC input shape:", X_pc_full_cpd.shape)

np.save(os.path.join(OUT_DIR, "cpd_pc_input.npy"), X_pc_full_cpd)
with open(os.path.join(OUT_DIR, "cpd_pc_col_names.json"), "w") as f:
    json.dump(col_names_cpd, f)
print("Saved.")

Smokers with valid CPD: 1631
Y_cpd mean: 26.35
Perfectly correlated SNPs to remove: 20
Final shortlist: 36 SNPs
PC input shape: (1631, 37)
Saved.
